In [8]:
import requests
import pandas as pd
import datetime
import json
import sqlite3
import traceback

# === 設定 ===
# DB_FILE = "/Users/lulutsai/Documents/NTPU_class/paper/code/mobile01_clawler/ntpu_paper.sqlite"
LOG_FILE = "eb_annotations.log"
DB_FILE = "D:/NTPU_class/paper/code/mobile01_clawler/ntpu_paper.sqlite"
API_TOKEN = 'sk-a94e722134454665b8fef72ddb349f37'
API_URL = "http://localhost:3000/api/v1/chat/completions"
LIMIT_COUNT = 1300 # ← 你想一次處理幾筆資料（可自行調整）
headers = {
    "Content-Type": "application/json",
    "Authorization": f"Bearer {API_TOKEN}"
}

# === 🔹 Log 工具 ===
def write_log(message: str, link_id=None, article_id=None, comment_id=None, status=None):
    timestamp = datetime.datetime.now().strftime("%Y-%m-%d %H:%M:%S")
    id_info = []
    if link_id:
        id_info.append(f"link_id={link_id}")
    if article_id:
        id_info.append(f"article_id={article_id}")
    if comment_id:
        id_info.append(f"comment_id={comment_id}")
    if status:
        id_info.append(f"status={status}")
    id_str = " | ".join(id_info)
    line = f"[{timestamp}] {message}"
    if id_str:
        line += f" | {id_str}"
    with open(LOG_FILE, "a", encoding="utf-8") as f:
        f.write(line + "\n")
    print(line)


# === 🔹 確保欄位存在 ===
def ensure_column(conn, table, column, definition):
    cur = conn.cursor()
    cur.execute(f"PRAGMA table_info({table});")
    columns = [c[1] for c in cur.fetchall()]
    if column not in columns:
        cur.execute(f"ALTER TABLE {table} ADD COLUMN {column} {definition};")
        conn.commit()
        write_log(f"🧱 已新增欄位 {table}.{column}")
    else:
        write_log(f"🔎 欄位 {column} 已存在，略過")


# === 🔹 建立 eb_annotations 資料表 ===
def ensure_eb_annotations_table(conn):
    cur = conn.cursor()
    cur.execute("""
    CREATE TABLE IF NOT EXISTS eb_annotations (
        id INTEGER PRIMARY KEY AUTOINCREMENT,
        link_id TEXT NOT NULL,
        article_id TEXT NOT NULL,
        comment_id TEXT,
        level INTEGER DEFAULT 1,
        created_at TEXT,
        strategy_power REAL,
        strategy_emotion REAL,
        strategy_blame REAL,
        score_fear REAL,
        score_obligation REAL,
        score_guilt REAL,
        pua_source TEXT,
        main_strategy TEXT,
        main_strategy_detail TEXT,
        confidence REAL,
        score_overall REAL,
        model_name TEXT,
        model_version TEXT,
        knowledge_base TEXT,
        FOREIGN KEY (article_id) REFERENCES articles(id),
        FOREIGN KEY (link_id) REFERENCES links(id)
    );
    """)
    conn.commit()
    write_log("✅ 確認資料表 eb_annotations 已存在")


# === 🔹 取得待分析文章 ===
def get_pending_articles(conn, limit):
    query = f"""
    SELECT id, link_id, content
    FROM articles
    WHERE content IS NOT NULL
      AND TRIM(content) != ''
      AND (pua_status IS NULL OR pua_status = 'pending')
    LIMIT {limit};
    """
    df = pd.read_sql_query(query, conn)
    write_log(f"📘 共讀取 {len(df)} 筆文章待分析（上限 {limit}）")
    return df


# === 🔹 更新文章狀態 ===
def update_article_status(conn, article_id, status, reason=None):
    cur = conn.cursor()
    cur.execute(
        "UPDATE articles SET pua_status = ?, pua_status_reason = ? WHERE id = ?;",
        (status, reason, article_id)
    )
    conn.commit()


# === 🔹 呼叫模型分析 ===
def analyze_text(text):
    data = {
        "model": "mistral:latest",
        "knowledge": ["pua_db"],
        "messages": [
            {
                "role": "system",
                "content": """你是一個以繁體中文回覆的情緒勒索分析助手。
請根據 Forward (1997) 與 陳燕諭 (2023) 對勒索策略的理論分類，
對給定文字進行情緒勒索分層評估，並輸出以下 JSON 結構：

{
  "strategy_power": 0~5,
  "strategy_emotion": 0~5,
  "strategy_blame": 0~5,
  "score_fear": 0~5,
  "score_obligation": 0~5,
  "score_guilt": 0~5,
  "pua_source": "family／partner／friend／workplace／online／self",
  "main_strategy": "power／emotion／blame／none",
  "main_strategy_detail": "更細緻的中文描述，例如『以哭訴方式威脅分手』或『混合型情緒操控』",
  "confidence": 0~1,
  "score_overall": 0~5,
  "is_eb_llm": 0 或 1：請判斷是否存在情緒勒索，若有情緒勒索為1，反之,
  "is_eb_llm_confidence": 0~1
}

定義說明：
- 0 代表該特徵完全沒有出現；
- 5 代表該特徵非常明顯；
- main_strategy 僅能是 power、emotion、blame、none 四類；
- 所有分數均為數值，請勿包含文字說明；
- 請務必只輸出純 JSON，不要任何解釋文字。
- is_eb_llm 為 0/1 的二元判定；is_eb_llm_confidence 為 0~1；
"""
            },
            {"role": "user", "content": f"請分析這段文字: {text}"}
        ]
    }

    res = requests.post(API_URL, headers=headers, json=data, timeout=90)
    res.raise_for_status()
    content = res.json()["choices"][0]["message"]["content"]

    try:
        result = json.loads(content)
    except json.JSONDecodeError:
        import re
        match = re.search(r'\{.*\}', content, re.S)
        if match:
            result = json.loads(match.group())
        else:
            raise ValueError(f"無法解析 JSON：{content}")
    return result


# === 🔹 分數檢查 ===
def validate_scores(result, link_id, article_id):
    for key in ["strategy_power", "strategy_emotion", "strategy_blame",
                "score_fear", "score_obligation", "score_guilt", "score_overall"]:
        if key in result:
            val = result[key]
            try:
                if not (0 <= float(val) <= 5):
                    write_log(f"⚠️ {key} 超出範圍 ({val})，已設為 None",
                              link_id=link_id, article_id=article_id)
                    result[key] = None
            except Exception:
                result[key] = None
    if "confidence" in result:
        try:
            if not (0 <= float(result["confidence"]) <= 1):
                write_log(f"⚠️ confidence 超出範圍 ({result['confidence']})，已設為 None",
                          link_id=link_id, article_id=article_id)
                result["confidence"] = None
        except Exception:
            result["confidence"] = None
    # is_eb_llm: 0/1
    if "is_eb_llm" in result:
        try:
            v = int(result["is_eb_llm"])
            result["is_eb_llm"] = v if v in (0, 1) else None
        except Exception:
            result["is_eb_llm"] = None

    # is_eb_llm_confidence: 0~1
    if "is_eb_llm_confidence" in result:
        try:
            v = float(result["is_eb_llm_confidence"])
            result["is_eb_llm_confidence"] = v if (0 <= v <= 1) else None
        except Exception:
            result["is_eb_llm_confidence"] = None
    return result

def calc_is_eb(result, overall_th=2.0, dim_th=2.0):
    """
    binary: 1=有情緒勒索跡象, 0=沒有
    規則可調：
    - score_overall >= overall_th => 1
    - 或 strategy / FOG 任一 >= dim_th => 1
    """
    def to_float(x):
        try:
            return float(x)
        except Exception:
            return None

    overall = to_float(result.get("score_overall"))
    dims = [
        to_float(result.get("strategy_power")),
        to_float(result.get("strategy_emotion")),
        to_float(result.get("strategy_blame")),
        to_float(result.get("score_fear")),
        to_float(result.get("score_obligation")),
        to_float(result.get("score_guilt")),
    ]

    if overall is not None and overall >= overall_th:
        return 1
    for v in dims:
        if v is not None and v >= dim_th:
            return 1
    return 0


# === 🔹 寫入分析結果 ===
def insert_analysis_result(conn, link_id, article_id, result):
    cur = conn.cursor()
    created_at = datetime.datetime.now().strftime("%Y-%m-%d %H:%M:%S")
    result = validate_scores(result, link_id, article_id)
    is_eb_rule = calc_is_eb(result, overall_th=2.0, dim_th=2.0)

    cur.execute("""
        INSERT INTO eb_annotations (
            link_id, article_id, comment_id, level, created_at,
            strategy_power, strategy_emotion, strategy_blame,
            score_fear, score_obligation, score_guilt,
            pua_source, main_strategy, main_strategy_detail,
            confidence, score_overall,
            is_eb_rule, is_eb_llm, is_eb_llm_confidence,
            model_name, model_version, knowledge_base
        ) VALUES (?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?);
    """, (
        link_id, article_id, result.get("comment_id"),
        result.get("level", 1), created_at,
        result.get("strategy_power"),
        result.get("strategy_emotion"),
        result.get("strategy_blame"),
        result.get("score_fear"),
        result.get("score_obligation"),
        result.get("score_guilt"),
        result.get("pua_source"),
        result.get("main_strategy"),
        result.get("main_strategy_detail") or "無明確描述",
        result.get("confidence"),
        result.get("score_overall"),
        is_eb_rule,
        result.get("is_eb_llm"),
        result.get("is_eb_llm_confidence"),
        result.get("model_name", "mistral"),
        result.get("model_version", "latest"),
        result.get("knowledge_base", "pua_db")
    ))
    conn.commit()


# === 🔸 主程式 ===
def main():
    conn = sqlite3.connect(DB_FILE)
    ensure_column(conn, "articles", "pua_status", "TEXT DEFAULT 'pending'")
    ensure_column(conn, "articles", "pua_status_reason", "TEXT")
    ensure_eb_annotations_table(conn)
    ensure_column(conn, "eb_annotations", "main_strategy_detail", "TEXT")
    ensure_column(conn, "eb_annotations", "is_eb_rule", "INTEGER DEFAULT 0")
    ensure_column(conn, "eb_annotations", "is_eb_llm", "INTEGER")
    ensure_column(conn, "eb_annotations", "is_eb_llm_confidence", "REAL")


    df = get_pending_articles(conn, LIMIT_COUNT)
    if df.empty:
        write_log("⚠️ 沒有待分析資料。")
        conn.close()
        return

    for idx, row in df.iterrows():
        link_id = row["link_id"]
        article_id = row["id"]
        text = row["content"]

        try:
            result = analyze_text(text)
            insert_analysis_result(conn, link_id, article_id, result)
            update_article_status(conn, article_id, "done")
            is_eb_rule = calc_is_eb(result, overall_th=2.0, dim_th=2.0)
            write_log(
                f"✅ DONE 第 {idx+1}/{len(df)} 筆 | overall={result.get('score_overall')} "
                f"| rule={is_eb_rule} | llm={result.get('is_eb_llm')}({result.get('is_eb_llm_confidence')}) "
                f"| main={result.get('main_strategy')} | detail={result.get('main_strategy_detail')}",
                link_id=link_id, article_id=article_id, status="done"
            )
        except Exception as e:
            reason = f"{type(e).__name__}: {str(e)[:100]}"
            update_article_status(conn, article_id, "error", reason)
            write_log(f"❌ ERROR 第 {idx+1}/{len(df)} 筆 | {reason}",
                      link_id=link_id, article_id=article_id, status="error")
            traceback.print_exc()

    conn.close()
    write_log("🎉 全部完成！")


# === 執行 ===
if __name__ == "__main__":
    main()

[2026-02-10 11:41:25] 🔎 欄位 pua_status 已存在，略過
[2026-02-10 11:41:25] 🔎 欄位 pua_status_reason 已存在，略過
[2026-02-10 11:41:25] ✅ 確認資料表 eb_annotations 已存在
[2026-02-10 11:41:25] 🔎 欄位 main_strategy_detail 已存在，略過
[2026-02-10 11:41:25] 🔎 欄位 is_eb_rule 已存在，略過
[2026-02-10 11:41:25] 🔎 欄位 is_eb_llm 已存在，略過
[2026-02-10 11:41:25] 🔎 欄位 is_eb_llm_confidence 已存在，略過
[2026-02-10 11:41:25] 📘 共讀取 1186 筆文章待分析（上限 1300）
[2026-02-10 11:41:40] ❌ ERROR 第 1/1186 筆 | ValueError: 無法解析 JSON： 這個文章提供了有關情感智能（Emotional Intelligence）和人際交往技巧的指南，以提高個人的情商能力。以下是主要內容：

1. **情感智能的基礎**: 提高情商 | link_id=68f9c2ffaf11375988d35b3b | article_id=68fd00bcdbeaa9ee9ad369bd | status=error


Traceback (most recent call last):
  File "C:\Users\USER\AppData\Local\Temp\ipykernel_2956\1599790101.py", line 158, in analyze_text
    result = json.loads(content)
  File "c:\Users\USER\.pyenv\pyenv-win\versions\3.10.5\lib\json\__init__.py", line 346, in loads
    return _default_decoder.decode(s)
  File "c:\Users\USER\.pyenv\pyenv-win\versions\3.10.5\lib\json\decoder.py", line 337, in decode
    obj, end = self.raw_decode(s, idx=_w(s, 0).end())
  File "c:\Users\USER\.pyenv\pyenv-win\versions\3.10.5\lib\json\decoder.py", line 355, in raw_decode
    raise JSONDecodeError("Expecting value", s, err.value) from None
json.decoder.JSONDecodeError: Expecting value: line 1 column 2 (char 1)

During handling of the above exception, another exception occurred:

Traceback (most recent call last):
  File "C:\Users\USER\AppData\Local\Temp\ipykernel_2956\1599790101.py", line 303, in main
    result = analyze_text(text)
  File "C:\Users\USER\AppData\Local\Temp\ipykernel_2956\1599790101.py", line 16

[2026-02-10 11:41:45] ✅ DONE 第 2/1186 筆 | overall=4 | rule=1 | llm=1(0.9) | main=emotion | detail=利用國安局長陳明通的言論來造成恐慌，嘗試讓選民認為只有投票民進黨才能保持台灣的安全 | link_id=68f9c472af1137024c861695 | article_id=68ff536edbeaa973acd48b5b | status=done
[2026-02-10 11:41:48] ✅ DONE 第 3/1186 筆 | overall=4 | rule=1 | llm=1(0.8) | main=power | detail=使用政治力量、民調和徵詢方式決定候選人 | link_id=68f9c472af1137024c861696 | article_id=68ff537cdbeaa973acd48b5c | status=done
[2026-02-10 11:41:51] ✅ DONE 第 4/1186 筆 | overall=3 | rule=1 | llm=0(1.0) | main=emotion | detail=Expressing desire and expectation towards a product release, showing anticipation | link_id=68f9c472af1137024c861697 | article_id=68ff538adbeaa973acd48b5d | status=done
[2026-02-10 11:41:57] ✅ DONE 第 5/1186 筆 | overall=3 | rule=1 | llm=1(0.9) | main=emotion | detail=使用情感層次高的語言（好感＋感嘆）來表達對台灣製造產品的喜愛，並暗示對於不支持台灣製造的人的負向看法 | link_id=68f9c472af1137024c861698 | article_id=68ff539bdbeaa973acd48b5e | status=done
[2026-02-10 11:42:02] ✅ DONE 第 6/1186 筆 | overall=3 | rule=1 | llm=

Traceback (most recent call last):
  File "C:\Users\USER\AppData\Local\Temp\ipykernel_2956\1599790101.py", line 158, in analyze_text
    result = json.loads(content)
  File "c:\Users\USER\.pyenv\pyenv-win\versions\3.10.5\lib\json\__init__.py", line 346, in loads
    return _default_decoder.decode(s)
  File "c:\Users\USER\.pyenv\pyenv-win\versions\3.10.5\lib\json\decoder.py", line 337, in decode
    obj, end = self.raw_decode(s, idx=_w(s, 0).end())
  File "c:\Users\USER\.pyenv\pyenv-win\versions\3.10.5\lib\json\decoder.py", line 355, in raw_decode
    raise JSONDecodeError("Expecting value", s, err.value) from None
json.decoder.JSONDecodeError: Expecting value: line 1 column 2 (char 1)

During handling of the above exception, another exception occurred:

Traceback (most recent call last):
  File "C:\Users\USER\AppData\Local\Temp\ipykernel_2956\1599790101.py", line 303, in main
    result = analyze_text(text)
  File "C:\Users\USER\AppData\Local\Temp\ipykernel_2956\1599790101.py", line 16

[2026-02-10 11:47:35] ✅ DONE 第 35/1186 筆 | overall=4.2 | rule=1 | llm=1(0.9) | main=emotion | detail=混合型情緒操控（使用情感和帶有恐懼感的語言） | link_id=68f9c516af113726b01d6e27 | article_id=68ff562fdbeaa973acd48b7d | status=done
[2026-02-10 11:47:44] ✅ DONE 第 36/1186 筆 | overall=3.6 | rule=1 | llm=1(0.7) | main=emotion | detail=使用情感哀求和迫力方式提出需求 | link_id=68f9c516af113726b01d6e28 | article_id=68ff563ddbeaa973acd48b7e | status=done
[2026-02-10 11:47:56] ✅ DONE 第 37/1186 筆 | overall=4 | rule=1 | llm=1(0.8) | main=emotion | detail=利用情緒勾引，並對嚴重後擊的方式施加壓力 | link_id=68f9c516af113726b01d6e29 | article_id=68ff5687dbeaa973acd48b7f | status=done
[2026-02-10 11:48:07] ✅ DONE 第 38/1186 筆 | overall=3.4 | rule=1 | llm=1(0.9) | main=emotion | detail=利用情感勾引、挑釁以及對方的壓力感覺 | link_id=68f9c516af113726b01d6e2a | article_id=68ff5696dbeaa973acd48b80 | status=done
[2026-02-10 11:48:19] ✅ DONE 第 39/1186 筆 | overall=3.6 | rule=1 | llm=1(0.8) | main=emotion | detail=混合型情緒操控（包含哀憐、義務感和自责） | link_id=68f9c516af113726b01d6e2b | article_id=6

...

[2026-02-10 15:52:47] ✅ DONE 第 1107/1186 筆 | overall=4 | rule=1 | llm=1(0.9) | main=emotion | detail=混合型情緒操控，使用哀愍、負責感以及罪悔來勒索 | link_id=69220a45dbeaa964a5706f01 | article_id=69232e4ddbeaa924d347f21c | status=done
[2026-02-10 15:52:58] ✅ DONE 第 1108/1186 筆 | overall=3 | rule=1 | llm=0(1.0) | main=emotion | detail=使用情感操作，表達疲勞、無聊等情感以求據取他人的好will，推薦電影 | link_id=69220b00dbeaa9654682a2a9 | article_id=69232e59dbeaa924d347f21d | status=done
[2026-02-10 15:53:11] ✅ DONE 第 1109/1186 筆 | overall=3 | rule=1 | llm=1(0.7) | main=emotion | detail=使用懷念和憶導情緒觸發感激之情，並暗示自己對娛樂廣場有一定程度的需要 | link_id=69220b00dbeaa9654682a2aa | article_id=69232e66dbeaa924d347f21e | status=done
[2026-02-10 15:53:22] ✅ DONE 第 1110/1186 筆 | overall=2 | rule=1 | llm=0(0.9) | main=emotion | detail=反思生命無常、自我進步與互動中避免傷害他人 | link_id=69220b00dbeaa9654682a2ab | article_id=69232e73dbeaa924d347f21f | status=done
[2026-02-10 15:53:34] ✅ DONE 第 1111/1186 筆 | overall=4 | rule=1 | llm=1(0.8) | main=emotion | detail=混合型情緒操控，包括哀愁、強制義務感和責任感 | link_i

Traceback (most recent call last):
  File "C:\Users\USER\AppData\Local\Temp\ipykernel_2956\1599790101.py", line 158, in analyze_text
    result = json.loads(content)
  File "c:\Users\USER\.pyenv\pyenv-win\versions\3.10.5\lib\json\__init__.py", line 346, in loads
    return _default_decoder.decode(s)
  File "c:\Users\USER\.pyenv\pyenv-win\versions\3.10.5\lib\json\decoder.py", line 337, in decode
    obj, end = self.raw_decode(s, idx=_w(s, 0).end())
  File "c:\Users\USER\.pyenv\pyenv-win\versions\3.10.5\lib\json\decoder.py", line 353, in raw_decode
    obj, end = self.scan_once(s, idx)
json.decoder.JSONDecodeError: Expecting ',' delimiter: line 10 column 61 (char 254)

During handling of the above exception, another exception occurred:

Traceback (most recent call last):
  File "C:\Users\USER\AppData\Local\Temp\ipykernel_2956\1599790101.py", line 303, in main
    result = analyze_text(text)
  File "C:\Users\USER\AppData\Local\Temp\ipykernel_2956\1599790101.py", line 163, in analyze_text


[2026-02-10 15:56:47] ✅ DONE 第 1126/1186 筆 | overall=4.2 | rule=1 | llm=1(0.8) | main=blame | detail=使用混合型情緒操控，包括責任感、嫌疑與恐懼 | link_id=69220b00dbeaa9654682a2bb | article_id=69232fb4dbeaa924d347f22f | status=done
[2026-02-10 15:57:05] ✅ DONE 第 1127/1186 筆 | overall=4 | rule=1 | llm=1(0.9) | main=blame | detail=使用情感和幾個例子來講述美國的歷史上惡行，並強調自己支持這部影片，同時提醒读者要注意他們所看到的線索 | link_id=69220b00dbeaa9654682a2bc | article_id=69232fc1dbeaa924d347f230 | status=done
[2026-02-10 15:57:17] ✅ DONE 第 1128/1186 筆 | overall=4 | rule=1 | llm=1(0.9) | main=emotion | detail=利用屁股感情和悲劇寒鬧引發怕惡之情，讓人感到憐憫和痛心 | link_id=69220b00dbeaa9654682a2bd | article_id=69232fcddbeaa924d347f231 | status=done
[2026-02-10 15:57:28] ✅ DONE 第 1129/1186 筆 | overall=3 | rule=1 | llm=0(1.0) | main=emotion | detail=使用情感化言論，以建立和溫和的環境，強調中囯之間的合作 | link_id=69220b00dbeaa9654682a2be | article_id=69232fdadbeaa924d347f232 | status=done
[2026-02-10 15:57:43] ✅ DONE 第 1130/1186 筆 | overall=3.6 | rule=1 | llm=0(1.0) | main=emotion | detail=使用喜歡的電影與遊戲導入情感，並引入

Traceback (most recent call last):
  File "C:\Users\USER\AppData\Roaming\Python\Python310\site-packages\urllib3\connectionpool.py", line 468, in _make_request
    six.raise_from(e, None)
  File "<string>", line 3, in raise_from
  File "C:\Users\USER\AppData\Roaming\Python\Python310\site-packages\urllib3\connectionpool.py", line 463, in _make_request
    httplib_response = conn.getresponse()
  File "c:\Users\USER\.pyenv\pyenv-win\versions\3.10.5\lib\http\client.py", line 1374, in getresponse
    response.begin()
  File "c:\Users\USER\.pyenv\pyenv-win\versions\3.10.5\lib\http\client.py", line 318, in begin
    version, status, reason = self._read_status()
  File "c:\Users\USER\.pyenv\pyenv-win\versions\3.10.5\lib\http\client.py", line 279, in _read_status
    line = str(self.fp.readline(_MAXLINE + 1), "iso-8859-1")
  File "c:\Users\USER\.pyenv\pyenv-win\versions\3.10.5\lib\socket.py", line 705, in readinto
    return self._sock.recv_into(b)
TimeoutError: timed out

During handling of the

[2026-02-10 16:03:06] ✅ DONE 第 1150/1186 筆 | overall=3.8 | rule=1 | llm=1(0.8) | main=emotion | detail=利用恐怖情感刺激觸動對方 | link_id=69220b00dbeaa9654682a2d3 | article_id=6923315ddbeaa924d347f247 | status=done
[2026-02-10 16:03:16] ✅ DONE 第 1151/1186 筆 | overall=0 | rule=0 | llm=0(1.0) | main=none | detail=無情緒勒索策略 | link_id=69220b00dbeaa9654682a2d4 | article_id=692331a6dbeaa924d347f248 | status=done
[2026-02-10 16:03:29] ✅ DONE 第 1152/1186 筆 | overall=3.5 | rule=1 | llm=1(0.9) | main=emotion | detail=Expressing dissatisfaction and disappointment, with a sense of expectation and entitlement towards the film's creators | link_id=69220b00dbeaa9654682a2d5 | article_id=692331b4dbeaa924d347f249 | status=done
[2026-02-10 16:03:40] ✅ DONE 第 1153/1186 筆 | overall=3 | rule=1 | llm=0(1.0) | main=emotion | detail=使用情緒創建社交連結 | link_id=69220b00dbeaa9654682a2d6 | article_id=692331c0dbeaa924d347f24a | status=done
[2026-02-10 16:03:53] ✅ DONE 第 1154/1186 筆 | overall=3 | rule=1 | llm=0(1.0) | main=emotion | de

In [2]:
import sqlite3
import pandas as pd

conn = sqlite3.connect("D:/NTPU_class/paper/code/mobile01_clawler/ntpu_paper.sqlite")

df = pd.read_sql_query("""
    SELECT article_id, COUNT(*) AS cnt
    FROM eb_evaluation
    GROUP BY article_id
    HAVING cnt > 1;
""", conn)

print(df)
print("🔍 有重複筆數：", len(df))

conn.close()


                    article_id  cnt
0     68fce9b6af1137205015fd06    9
1     68fce9f3af11377624f11eda   15
2     68fcea01af11377624f11edb    9
3     68fcea6eaf11379f9c929394    9
4     68fcea81af11379f9c929395    9
...                        ...  ...
1284  6923413cdbeaa944f0dd8399    9
1285  69234149dbeaa944f0dd839a    2
1286  69234156dbeaa944f0dd839b    4
1287  69234163dbeaa944f0dd839c    9
1288  69234171dbeaa944f0dd839d    9

[1289 rows x 2 columns]
🔍 有重複筆數： 1289


In [4]:
import re
import requests
import time
import json

LIMIT_COUNT =  500
def analyze_article_and_comment(article_text, comment_text, comment_id=None):
    data = {
        "model": "mistral:latest",
        "knowledge": ["pua_db", "healthy_communication_db", "social_commentary_advice_db"],
        "messages": [
                    {
                        "role": "system",
                        "content": """你是一個以繁體中文回覆的情緒勒索分析助手。
❗❗極度重要（請務必嚴格遵守，優先度最高）：
- 若評論中「沒有明確的操控、威脅、施壓、罪惡感誘發、恐懼引導、責任捆綁、指責、情緒綁架」，請將所有分數評為 0。
- 若評論內容屬於以下任一類型，也請評為 0：
  • 善意建議
  • 中立敘述
  • 理性討論
  • 情緒抒發但未要求對方負責
  • 一般社群語言、玩笑、生活分享
  • 批評但未包含控制或操作
- 除非評論中「明確存在」勒索策略，否則一律視為無勒索（即分數=0，main_strategy="none"）。
- 請勿因為語氣、情緒、抱怨、不滿，而推測或想像可能存在的勒索動機。
- 給出最低分（0）是最常見也是最正確的狀態。


📌 僅在出現「明確的施壓或操控語句」時，才可給 1~5 分。

分析規則：
- 本任務的分析主體是評論（comment）。
- 主文（article）僅提供上下文理解，不應被評分。
- 所有策略與分數僅針對評論內容進行判斷。
請根據 Forward (1997) 與 陳燕諭 (2023) 對勒索策略的理論分類，若評論中完全沒有操控成分，請將所有分數設為 0。
對給定文字進行情緒勒索分層評估，並輸出以下 JSON 結構：

{
"strategy_power": 0~5,
"strategy_emotion": 0~5,
"strategy_blame": 0~5,
"score_fear": 0~5,
"score_obligation": 0~5,
"score_guilt": 0~5,
"pua_source": "family／partner／friend／workplace／online／self",
"main_strategy": "power／emotion／blame／none",
"main_strategy_detail": "更細緻的中文描述，例如『以哭訴方式威脅分手』或『混合型情緒操控』",
"confidence": 0~1,
"score_overall": 0~5
}

補充說明：
- pua_source 請選擇「最符合的一個主要來源」。
- 若語意涉及多個來源，請選擇你判斷中影響最核心者。
- 若無法明確判定，請選擇 online。
                                """
                                                            },
                    {
                        "role": "user",
                        "content": f"""請分析以下內容，並將「評論(comment)」視為主要分析對象：

【主文（僅作為上下文，取300字）】
{article_text}

【評論（主要分析目標）】
{comment_text}""".strip()
                    }
                ]
    }
    print(data)

    # 補充說明是20260113加上得

    res = requests.post(API_URL, headers=headers, json=data, timeout=90)
    print(res.status_code)
    print(res.text)
    content = res.json()["choices"][0]["message"]["content"]

    # 避免模型包文字 → 正常處理
    try:
        result = json.loads(content)
    except:
        match = re.search(r'\{.*\}', content, re.S)
        result = json.loads(match.group()) if match else {}

    # 自動補上 comment_id
    if comment_id and "comment_id" not in result:
        result["comment_id"] = comment_id

    return result

def get_pending_comments(conn, limit):
    query = f"""
    SELECT 
        c.id AS comment_id,
        c.comment_text AS comment_text,
        a.id AS article_id,
        a.link_id AS link_id,
        a.content AS article_text
    FROM article_comments c
    JOIN articles a 
        ON c.article_id = a.id
    WHERE c.comment_text IS NOT NULL
      AND TRIM(c.comment_text) != ''
      AND (c.pua_status IS NULL OR c.pua_status = 'pending')
    LIMIT {limit};
    """
    df = pd.read_sql_query(query, conn)
    write_log(f"📘 共讀取 {len(df)} 筆評論待分析（上限 {limit}）")
    return df


def update_comment_status(conn, comment_id, status, reason=None):
    cur = conn.cursor()
    cur.execute(
        "UPDATE article_comments SET pua_status = ?, pua_status_reason = ? WHERE id = ?;",
        (status, reason, comment_id)
    )
    conn.commit()
    

def insert_analysis_result(conn, link_id, article_id, result):
    cur = conn.cursor()
    created_at = datetime.datetime.now().strftime("%Y-%m-%d %H:%M:%S")
    result = validate_scores(result, link_id, article_id)

    cur.execute("""
        INSERT INTO eb_evaluation (
            link_id, article_id, comment_id, level, created_at,
            strategy_power, strategy_emotion, strategy_blame,
            score_fear, score_obligation, score_guilt,
            pua_source, main_strategy, main_strategy_detail,
            confidence, score_overall,
            model_name, model_version, knowledge_base
        ) VALUES (?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?);
    """, (
        link_id, article_id, result.get("comment_id"),
        result.get("level", 2), created_at,
        result.get("strategy_power"),
        result.get("strategy_emotion"),
        result.get("strategy_blame"),
        result.get("score_fear"),
        result.get("score_obligation"),
        result.get("score_guilt"),
        result.get("pua_source"),
        result.get("main_strategy"),
        result.get("main_strategy_detail") or "無明確描述",
        result.get("confidence"),
        result.get("score_overall"),
        result.get("model_name", "mistral"),
        result.get("model_version", "latest"),
        result.get("knowledge_base", "pua_db, healthy_communication_db, social_commentary_advice_db")
    ))
    conn.commit()


# === 🔸 主程式 ===
def main():
    conn = sqlite3.connect(DB_FILE)
    ensure_column(conn, "articles", "pua_status", "TEXT DEFAULT 'pending'")
    ensure_column(conn, "articles", "pua_status_reason", "TEXT")
    ensure_eb_evaluation_table(conn)
    ensure_column(conn, "eb_evaluation", "main_strategy_detail", "TEXT")
    # ⭐ 新增：評論表也要有狀態欄位
    ensure_column(conn, "article_comments", "pua_status", "TEXT DEFAULT 'pending'")
    ensure_column(conn, "article_comments", "pua_status_reason", "TEXT")

    df = get_pending_comments(conn, LIMIT_COUNT)
    if df.empty:
        write_log("⚠️ 沒有待分析評論。")
        conn.close()
        return

    for idx, row in df.iterrows():
        article_text = row["article_text"]
        comment_text = row["comment_text"]
        comment_id = row["comment_id"]
        article_id = row["article_id"]
        link_id = row["link_id"]

        try:
            result = analyze_article_and_comment(article_text, comment_text, comment_id)

            insert_analysis_result(conn, link_id, article_id, result)

            update_comment_status(conn, comment_id, "done")

            write_log(
                f"✅ DONE 第 {idx+1}/{len(df)} 筆 | overall={result.get('score_overall')} "
                f"| main={result.get('main_strategy')} | detail={result.get('main_strategy_detail')}",
                link_id=link_id, article_id=article_id, comment_id=comment_id, status="done"
            )

        except Exception as e:
            reason = f"{type(e).__name__}: {str(e)[:100]}"
            update_comment_status(conn, comment_id, "error", reason)

            write_log(
                f"❌ ERROR 第 {idx+1}/{len(df)} 筆 | {reason}",
                link_id=link_id, article_id=article_id, comment_id=comment_id, status="error"
            )
            traceback.print_exc()
        time.sleep(0.5)

    conn.close()
    write_log("🎉 全部完成！")


# === 執行 ===
if __name__ == "__main__":
    main()

[2026-02-03 10:20:41] 🔎 欄位 pua_status 已存在，略過
[2026-02-03 10:20:41] 🔎 欄位 pua_status_reason 已存在，略過
[2026-02-03 10:20:41] ✅ 確認資料表 eb_evaluation 已存在
[2026-02-03 10:20:41] 🔎 欄位 main_strategy_detail 已存在，略過
[2026-02-03 10:20:41] 🔎 欄位 pua_status 已存在，略過
[2026-02-03 10:20:41] 🔎 欄位 pua_status_reason 已存在，略過
[2026-02-03 10:20:41] 📘 共讀取 48 筆評論待分析（上限 500）
{'model': 'mistral:latest', 'knowledge': ['pua_db', 'healthy_communication_db', 'social_commentary_advice_db'], 'messages': [{'role': 'system', 'content': '你是一個以繁體中文回覆的情緒勒索分析助手。\n❗❗極度重要（請務必嚴格遵守，優先度最高）：\n- 若評論中「沒有明確的操控、威脅、施壓、罪惡感誘發、恐懼引導、責任捆綁、指責、情緒綁架」，請將所有分數評為 0。\n- 若評論內容屬於以下任一類型，也請評為 0：\n  • 善意建議\n  • 中立敘述\n  • 理性討論\n  • 情緒抒發但未要求對方負責\n  • 一般社群語言、玩笑、生活分享\n  • 批評但未包含控制或操作\n- 除非評論中「明確存在」勒索策略，否則一律視為無勒索（即分數=0，main_strategy="none"）。\n- 請勿因為語氣、情緒、抱怨、不滿，而推測或想像可能存在的勒索動機。\n- 給出最低分（0）是最常見也是最正確的狀態。\n\n\n📌 僅在出現「明確的施壓或操控語句」時，才可給 1~5 分。\n\n分析規則：\n- 本任務的分析主體是評論（comment）。\n- 主文（article）僅提供上下文理解，不應被評分。\n- 所有策略與分數僅針對評論內容進行判斷。\n請根據 Forward (1997) 與 陳燕諭 (2023) 對勒索策略的理論分類，

In [24]:
import sqlite3
import json
import pandas as pd

DB_FILE = "D:/NTPU_class/paper/code/mobile01_clawler/ntpu_paper.sqlite"

def get_comment_and_scores_as_dict(comment_id):
    conn = sqlite3.connect(DB_FILE)

    # 📝 抓評論
    df_comment = pd.read_sql_query("""
        SELECT 
            id AS comment_id,
            comment_text,
            article_id,
            link_id
        FROM article_comments
        WHERE id = ?
    """, conn, params=(comment_id,))

    if df_comment.empty:
        conn.close()
        return {"error": f"comment_id {comment_id} 不存在"}

    comment = df_comment.to_dict(orient="records")[0]

    # 📊 抓評分
    df_scores = pd.read_sql_query("""
        SELECT
            id AS eval_id,
            created_at,
            strategy_power,
            strategy_emotion,
            strategy_blame,
            score_fear,
            score_obligation,
            score_guilt,
            pua_source,
            main_strategy,
            main_strategy_detail,
            confidence,
            score_overall,
            model_name,
            model_version,
            knowledge_base
        FROM eb_evaluation
        WHERE comment_id = ?
        ORDER BY created_at ASC
    """, conn, params=(comment_id,))

    conn.close()

    scores = df_scores.to_dict(orient="records")

    return {
        "comment": comment,
        "evaluations": scores
    }


# 🧪 測試
if __name__ == "__main__":
    cid = "6923bf32af113796ec890f8b"   # ← 改成你的 comment_id
    data = get_comment_and_scores_as_dict(cid)
    print(json.dumps(data, ensure_ascii=False, indent=2))

{
  "comment": {
    "comment_id": "6923bf32af113796ec890f8b",
    "comment_text": "我覺得既然有發現問題，可以去聽聽看心理諮商師的說法，或許會找到解法也不一定～如果擔心費用現在政府也有補助方案可以詢問看看！",
    "article_id": "68fce9b6af1137205015fd06",
    "link_id": "68f87edcaf1137700cb26e94"
  },
  "evaluations": [
    {
      "eval_id": 1410,
      "created_at": "2025-12-03 10:20:20",
      "strategy_power": 0.0,
      "strategy_emotion": 4.0,
      "strategy_blame": 3.0,
      "score_fear": 2.0,
      "score_obligation": 1.0,
      "score_guilt": 1.0,
      "pua_source": "family",
      "main_strategy": "emotion",
      "main_strategy_detail": "情緒操控，使用情感哀求、指責、壓力語言等方式來勒索",
      "confidence": 1.0,
      "score_overall": 3.0,
      "model_name": "mistral",
      "model_version": "latest",
      "knowledge_base": "pua_db, healthy_communication_db, social_commentary_advice_db"
    }
  ]
}


### 檢查多選的欄位抓出來讓他們重跑

In [4]:
import sqlite3
from datetime import datetime

ALLOWED = ("emotion","power","blame","obligation","information","none")

def mark_dirty_comments_pending(conn: sqlite3.Connection) -> int:
    cur = conn.cursor()

    # 1) 找出 eb_evaluation 裡 main_strategy 髒的 comment_id
    cur.execute(f"""
        SELECT DISTINCT comment_id
        FROM eb_evaluation
        WHERE comment_id IS NOT NULL
          AND (
                main_strategy IS NULL
             OR TRIM(main_strategy) = ''
             OR main_strategy LIKE '%/%'
             OR main_strategy NOT IN {ALLOWED}
          )
    """)
    dirty_comment_ids = [r[0] for r in cur.fetchall()]
    print(dirty_comment_ids)
    print(len(dirty_comment_ids))

    if not dirty_comment_ids:
        return 0

    # 2) 把這些 comment 改回 pending
    placeholders = ",".join(["?"] * len(dirty_comment_ids))
    cur.execute(
        f"""
        UPDATE article_comments
        SET pua_status = 'pending',
            pua_status_reason = 'rerun_dirty_main_strategy'
        WHERE id IN ({placeholders})
        """,
        dirty_comment_ids
    )

    conn.commit()
    return cur.rowcount

conn = sqlite3.connect("D:/NTPU_class/paper/code/mobile01_clawler/ntpu_paper.sqlite")
dirty_count = mark_dirty_comments_pending(conn)

['6923bf32af113796ec890fd0', '6923bf32af113796ec890ff6', '6923bf32af113796ec890ffd', '6923bf32af113796ec89100e', '6923bf32af113796ec891023', '6923bf32af113796ec89102e', '6923bf32af113796ec8910b6', '6923bf32af113796ec891149', '6923bf32af113796ec89114d', '6923bf32af113796ec891155', '6923bf32af113796ec89115f', '6923bf32af113796ec89116d', '6923bf32af113796ec89116f', '6923bf32af113796ec891170', '6923bf32af113796ec891171', '6923bf32af113796ec891172', '6923bf32af113796ec891173', '6923bf32af113796ec891174', '6923bf32af113796ec891175', '6923bf32af113796ec891176', '6923bf32af113796ec891182', '6923bf32af113796ec891193', '6923bf32af113796ec891197', '6923bf32af113796ec8911f2', '6923bf32af113796ec8911fc', '6923bf32af113796ec8911ff', '6923bf32af113796ec891217', '6923bf32af113796ec891242', '6923bf32af113796ec891249', '6923bf32af113796ec891253', '6923bf32af113796ec89125b', '6923bf32af113796ec89129e', '6923bf32af113796ec8912a3', '6923bf32af113796ec8912c9', '6923bf32af113796ec8912d6', '6923bf32af113796ec